# Xcapit FHE-ML Platform - Gobierno: Asignacion de Recursos

## Caso de Uso: Consorcio Inter-Provincial para Optimizar Servicios Publicos

### El Problema
Tres provincias argentinas quieren optimizar la asignacion de recursos, pero:
- Cada provincia tiene datos sensibles de ciudadanos
- No pueden compartir datos personales entre jurisdicciones
- Sin colaboracion, cada provincia "reinventa la rueda"

### La Solucion: FHE para Gobierno
Permite analizar demanda de servicios a nivel nacional **sin que ninguna provincia vea datos de otra**.

---

## Flujo del Demo (Paso a Paso)

```
+------------------+     +------------------+     +------------------+
|   PASO 1-2       |     |   PASO 3-4       |     |   PASO 5-6       |
|   Setup +        | --> |   Datos de       | --> |   Segmentacion   |
|   Config         |     |   Ciudadanos     |     |   Poblacion      |
+------------------+     +------------------+     +------------------+
         |                        |                        |
         v                        v                        v
+------------------+     +------------------+     +------------------+
|   PASO 7         |     |   PASO 8         |     |   PASO 9         |
|   Modelo         | --> |   Optimizacion   | --> |   Dashboard      |
|   Predictivo     |     |   Recursos       |     |   Indicadores    |
+------------------+     +------------------+     +------------------+
```

### Que aprenderas:
- Como segmentar poblacion con K-Means sobre datos encriptados
- Como predecir demanda de servicios publicos
- Como optimizar asignacion de presupuesto
- Como generar dashboards con estadisticas agregadas

---

### Escenario Simulado
| Provincia | Codigo | Registros | Presupuesto |
|-----------|--------|-----------|-------------|
| Buenos Aires | BA | 10,000 | $500M |
| Cordoba | CBA | 5,000 | $250M |
| Santa Fe | SF | 5,000 | $250M |

### Servicios Analizados
- **Salud**: Visitas medicas, hospitalizaciones
- **Educacion**: Inscripciones, becas
- **Asistencia Social**: Subsidios, programas
- **Vivienda**: Subsidios habitacionales

### Cumplimiento
- **Ley 25.326**: Proteccion de datos personales
- **ISO 27001**: Seguridad de la informacion
- **NIST**: Estandares de ciberseguridad

## 1. Setup e Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))

import numpy as np
import pandas as pd
import hashlib
from datetime import datetime
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, r2_score, silhouette_score

print("Xcapit FHE-ML Platform - Government Demo")
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)
print("\nCumplimiento: Ley 25.326, ISO 27001, NIST")

## 2. Configuracion del Dataset Government

In [ ]:
# Configuracion del dataset
GOVERNMENT_CONFIG = {
    "n_samples": 20000,
    "provinces": [
        {"name": "Buenos Aires", "code": "BA", "samples": 10000, "budget_millions": 500},
        {"name": "Cordoba", "code": "CBA", "samples": 5000, "budget_millions": 250},
        {"name": "Santa Fe", "code": "SF", "samples": 5000, "budget_millions": 250},
    ],
    "services": [
        "health", "education", "social_assistance", "housing", "employment"
    ],
    "features": [
        # Datos demograficos anonimizados
        {"name": "age_group", "type": "category", "values": ["0-17", "18-34", "35-54", "55-64", "65+"]},
        {"name": "household_size", "type": "int", "min": 1, "max": 8},
        {"name": "income_bracket", "type": "category", "values": ["low", "lower_middle", "middle", "upper_middle", "high"]},
        {"name": "education_level", "type": "category", "values": ["none", "primary", "secondary", "tertiary", "university"]},
        {"name": "employment_status", "type": "category", "values": ["employed", "unemployed", "retired", "student", "informal"]},
        
        # Uso de servicios (ultimos 12 meses)
        {"name": "health_visits", "type": "int", "min": 0, "max": 24},
        {"name": "education_enrollment", "type": "bool", "true_ratio": 0.35},
        {"name": "social_assistance", "type": "bool", "true_ratio": 0.15},
        {"name": "housing_subsidy", "type": "bool", "true_ratio": 0.08},
        
        # Ubicacion (agregada)
        {"name": "urban_rural", "type": "category", "values": ["urban", "suburban", "rural"]},
        {"name": "zone_vulnerability_index", "type": "float", "min": 0, "max": 1},
    ]
}

print("Configuracion Government:")
print(f"  Total registros: {GOVERNMENT_CONFIG['n_samples']:,}")
print(f"  Provincias participantes: {len(GOVERNMENT_CONFIG['provinces'])}")
print(f"  Servicios analizados: {len(GOVERNMENT_CONFIG['services'])}")
print(f"  Features: {len(GOVERNMENT_CONFIG['features'])}")

In [ ]:
def generate_government_data(config: dict, seed: int = 42) -> pd.DataFrame:
    """Genera datos sinteticos de ciudadanos para analisis de servicios."""
    np.random.seed(seed)
    n = config["n_samples"]
    
    df = pd.DataFrame()
    
    # Asignar provincias
    province_labels = []
    province_codes = []
    for prov in config["provinces"]:
        province_labels.extend([prov["name"]] * prov["samples"])
        province_codes.extend([prov["code"]] * prov["samples"])
    
    df['province'] = province_labels[:n]
    df['province_code'] = province_codes[:n]
    
    # ID anonimizado
    df['citizen_id'] = [f"CIT-{hashlib.sha256(str(i).encode()).hexdigest()[:10].upper()}" for i in range(n)]
    
    # Grupo etario
    age_groups = ["0-17", "18-34", "35-54", "55-64", "65+"]
    age_probs = [0.22, 0.25, 0.28, 0.12, 0.13]  # Distribucion argentina aprox
    df['age_group'] = np.random.choice(age_groups, n, p=age_probs)
    
    # Tamano del hogar
    df['household_size'] = np.random.poisson(2.5, n).clip(1, 8)
    
    # Nivel de ingreso
    income_levels = ["low", "lower_middle", "middle", "upper_middle", "high"]
    income_probs = [0.20, 0.25, 0.30, 0.15, 0.10]
    df['income_bracket'] = np.random.choice(income_levels, n, p=income_probs)
    
    # Nivel educativo (correlacionado con edad e ingreso)
    education_levels = ["none", "primary", "secondary", "tertiary", "university"]
    df['education_level'] = np.random.choice(education_levels, n, p=[0.05, 0.20, 0.35, 0.25, 0.15])
    
    # Estado de empleo
    employment_status = ["employed", "unemployed", "retired", "student", "informal"]
    df['employment_status'] = np.random.choice(employment_status, n, p=[0.45, 0.10, 0.15, 0.15, 0.15])
    
    # Visitas de salud (mayores y ninos mas visitas)
    df['health_visits'] = np.random.poisson(4, n)
    df.loc[df['age_group'].isin(['0-17', '65+']), 'health_visits'] += np.random.poisson(3, (df['age_group'].isin(['0-17', '65+'])).sum())
    df['health_visits'] = df['health_visits'].clip(0, 24)
    
    # Inscripcion educativa
    df['education_enrollment'] = False
    df.loc[df['age_group'] == '0-17', 'education_enrollment'] = np.random.random((df['age_group'] == '0-17').sum()) < 0.95
    df.loc[df['age_group'] == '18-34', 'education_enrollment'] = np.random.random((df['age_group'] == '18-34').sum()) < 0.25
    
    # Asistencia social (correlacionada con ingreso)
    df['social_assistance'] = False
    df.loc[df['income_bracket'] == 'low', 'social_assistance'] = np.random.random((df['income_bracket'] == 'low').sum()) < 0.6
    df.loc[df['income_bracket'] == 'lower_middle', 'social_assistance'] = np.random.random((df['income_bracket'] == 'lower_middle').sum()) < 0.2
    
    # Subsidio de vivienda
    df['housing_subsidy'] = (df['income_bracket'].isin(['low', 'lower_middle'])) & (np.random.random(n) < 0.15)
    
    # Urbano/Rural
    urban_types = ["urban", "suburban", "rural"]
    df['urban_rural'] = np.random.choice(urban_types, n, p=[0.65, 0.25, 0.10])
    
    # Indice de vulnerabilidad de zona
    df['zone_vulnerability_index'] = np.random.beta(2, 5, n)
    df.loc[df['income_bracket'] == 'low', 'zone_vulnerability_index'] += 0.2
    df.loc[df['urban_rural'] == 'rural', 'zone_vulnerability_index'] += 0.1
    df['zone_vulnerability_index'] = df['zone_vulnerability_index'].clip(0, 1).round(3)
    
    # Calcular demanda de servicios (target para prediccion)
    df['service_demand_score'] = (
        df['health_visits'] * 0.3 +
        df['education_enrollment'].astype(int) * 2 +
        df['social_assistance'].astype(int) * 3 +
        df['housing_subsidy'].astype(int) * 4 +
        df['zone_vulnerability_index'] * 5
    ).round(2)
    
    return df

# Generar datos
df_citizens = generate_government_data(GOVERNMENT_CONFIG)

print("\nDataset Generado:")
print(f"  Shape: {df_citizens.shape}")
print(f"\nDistribucion por provincia:")
print(df_citizens.groupby('province').agg({
    'citizen_id': 'count',
    'social_assistance': 'mean',
    'zone_vulnerability_index': 'mean'
}).round(3))

In [ ]:
# Vista previa (anonimizada)
print("Vista previa de registros (anonimizado):")
print("="*90)
display_cols = ['citizen_id', 'province_code', 'age_group', 'income_bracket', 
                'health_visits', 'social_assistance', 'zone_vulnerability_index']
df_citizens[display_cols].head(10)

## 3. Analisis Agregado por Provincia

Estadisticas agregadas que NO revelan datos individuales.

In [ ]:
print("ANALISIS AGREGADO POR PROVINCIA")
print("=" * 70)

for prov in GOVERNMENT_CONFIG['provinces']:
    prov_data = df_citizens[df_citizens['province'] == prov['name']]
    
    stats = {
        'registros': len(prov_data),
        'asistencia_social_pct': prov_data['social_assistance'].mean() * 100,
        'subsidio_vivienda_pct': prov_data['housing_subsidy'].mean() * 100,
        'inscripcion_educativa_pct': prov_data['education_enrollment'].mean() * 100,
        'visitas_salud_promedio': prov_data['health_visits'].mean(),
        'vulnerabilidad_promedio': prov_data['zone_vulnerability_index'].mean(),
        'demanda_promedio': prov_data['service_demand_score'].mean()
    }
    
    print(f"\n{prov['name']} ({prov['code']}) - Presupuesto: ${prov['budget_millions']}M")
    print("-" * 50)
    print(f"   Registros: {stats['registros']:,}")
    print(f"   Asistencia social: {stats['asistencia_social_pct']:.1f}%")
    print(f"   Subsidio vivienda: {stats['subsidio_vivienda_pct']:.1f}%")
    print(f"   Inscripcion educativa: {stats['inscripcion_educativa_pct']:.1f}%")
    print(f"   Visitas salud (promedio): {stats['visitas_salud_promedio']:.1f}")
    print(f"   Indice vulnerabilidad: {stats['vulnerabilidad_promedio']:.3f}")
    print(f"   Score demanda servicios: {stats['demanda_promedio']:.2f}")

## 4. Segmentacion de Poblacion (K-Means sobre datos encriptados)

In [ ]:
print("SEGMENTACION DE POBLACION (K-Means FHE)")
print("=" * 60)

# Preparar features para clustering
cluster_features = ['household_size', 'health_visits', 'zone_vulnerability_index', 'service_demand_score']

# Encoding de categoricas
le_income = LabelEncoder()
le_age = LabelEncoder()

df_cluster = df_citizens.copy()
df_cluster['income_encoded'] = le_income.fit_transform(df_cluster['income_bracket'])
df_cluster['age_encoded'] = le_age.fit_transform(df_cluster['age_group'])

X_cluster = df_cluster[cluster_features + ['income_encoded', 'age_encoded']].values

# Normalizar
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# K-Means
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_cluster_scaled)

df_citizens['segment'] = clusters

silhouette = silhouette_score(X_cluster_scaled, clusters)
print(f"Numero de segmentos: {n_clusters}")
print(f"Silhouette score: {silhouette:.4f}")

In [ ]:
# Caracterizar segmentos
print("\nCARACTERIZACION DE SEGMENTOS")
print("=" * 70)

segment_names = {
    0: "Familias vulnerables",
    1: "Jovenes estudiantes",
    2: "Adultos mayores",
    3: "Clase media trabajadora",
    4: "Hogares estables"
}

for seg in range(n_clusters):
    seg_data = df_citizens[df_citizens['segment'] == seg]
    
    print(f"\nSegmento {seg}: {segment_names.get(seg, 'Sin nombre')}")
    print("-" * 50)
    print(f"   Tamano: {len(seg_data):,} ({len(seg_data)/len(df_citizens)*100:.1f}%)")
    print(f"   Vulnerabilidad promedio: {seg_data['zone_vulnerability_index'].mean():.3f}")
    print(f"   Asistencia social: {seg_data['social_assistance'].mean()*100:.1f}%")
    print(f"   Demanda servicios: {seg_data['service_demand_score'].mean():.2f}")
    print(f"   Grupo etario mas comun: {seg_data['age_group'].mode().values[0]}")
    print(f"   Ingreso mas comun: {seg_data['income_bracket'].mode().values[0]}")

## 5. Modelo Predictivo de Demanda de Servicios

In [ ]:
print("MODELO PREDICTIVO DE DEMANDA")
print("=" * 60)

# Preparar features
feature_cols = ['household_size', 'health_visits', 'zone_vulnerability_index',
                'income_encoded', 'age_encoded']

# Encoding adicional
le_urban = LabelEncoder()
le_emp = LabelEncoder()

df_model = df_citizens.copy()
df_model['income_encoded'] = le_income.transform(df_model['income_bracket'])
df_model['age_encoded'] = le_age.transform(df_model['age_group'])
df_model['urban_encoded'] = le_urban.fit_transform(df_model['urban_rural'])
df_model['employment_encoded'] = le_emp.fit_transform(df_model['employment_status'])
df_model['social_int'] = df_model['social_assistance'].astype(int)
df_model['housing_int'] = df_model['housing_subsidy'].astype(int)

all_features = feature_cols + ['urban_encoded', 'employment_encoded', 'social_int', 'housing_int']

X = df_model[all_features].values
y = df_model['service_demand_score'].values

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Datos de entrenamiento: {len(X_train):,}")
print(f"Datos de prueba: {len(X_test):,}")

# Entrenar modelo
print("\nEntrenando Gradient Boosting Regressor...")
model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluar
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nEntrenamiento completado!")
print(f"MAE: {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

## 6. Optimizacion de Asignacion de Recursos

In [ ]:
print("OPTIMIZACION DE ASIGNACION DE RECURSOS")
print("=" * 70)

# Calcular demanda total por provincia y servicio
allocation = []

for prov in GOVERNMENT_CONFIG['provinces']:
    prov_data = df_citizens[df_citizens['province'] == prov['name']]
    
    # Calcular necesidades por servicio
    health_need = prov_data['health_visits'].sum() * 100  # $ por visita
    education_need = prov_data['education_enrollment'].sum() * 5000  # $ por estudiante
    social_need = prov_data['social_assistance'].sum() * 2000  # $ por beneficiario
    housing_need = prov_data['housing_subsidy'].sum() * 10000  # $ por subsidio
    
    total_need = health_need + education_need + social_need + housing_need
    budget = prov['budget_millions'] * 1_000_000
    
    allocation.append({
        'province': prov['name'],
        'budget': budget,
        'health_need': health_need,
        'education_need': education_need,
        'social_need': social_need,
        'housing_need': housing_need,
        'total_need': total_need,
        'coverage': min(budget / total_need * 100, 100) if total_need > 0 else 100
    })

df_allocation = pd.DataFrame(allocation)

for _, row in df_allocation.iterrows():
    print(f"\n{row['province']}:")
    print(f"   Presupuesto: ${row['budget']/1_000_000:.0f}M")
    print(f"   Necesidad total estimada: ${row['total_need']/1_000_000:.1f}M")
    print(f"   Cobertura proyectada: {row['coverage']:.1f}%")
    print(f"   Distribucion recomendada:")
    for service in ['health', 'education', 'social', 'housing']:
        need = row[f'{service}_need']
        pct = need / row['total_need'] * 100 if row['total_need'] > 0 else 0
        allocated = min(need, row['budget'] * pct / 100)
        print(f"      {service.capitalize():12} ${allocated/1_000_000:>6.2f}M ({pct:.1f}%)")

In [ ]:
# Recomendaciones basadas en datos
print("\nRECOMENDACIONES BASADAS EN DATOS")
print("=" * 60)

for seg in range(n_clusters):
    seg_data = df_citizens[df_citizens['segment'] == seg]
    vuln = seg_data['zone_vulnerability_index'].mean()
    demand = seg_data['service_demand_score'].mean()
    
    print(f"\nSegmento {seg} ({segment_names.get(seg, 'N/A')}):")
    
    recommendations = []
    
    if vuln > 0.5:
        recommendations.append("Priorizar programas de asistencia directa")
    
    if seg_data['social_assistance'].mean() > 0.3:
        recommendations.append("Evaluar transicion a programas de empleo")
    
    if seg_data['age_group'].mode().values[0] == '65+':
        recommendations.append("Fortalecer servicios de salud geriatrica")
    
    if seg_data['age_group'].mode().values[0] == '0-17':
        recommendations.append("Asegurar cobertura educativa completa")
    
    if seg_data['housing_subsidy'].mean() > 0.1:
        recommendations.append("Evaluar programas de vivienda social")
    
    if not recommendations:
        recommendations.append("Mantener programas actuales")
    
    for rec in recommendations:
        print(f"   - {rec}")

## 7. Dashboard de Indicadores

In [ ]:
print("DASHBOARD DE INDICADORES DEL CONSORCIO")
print("=" * 70)

print("""
┌─────────────────────────────────────────────────────────────────────┐
│                    CONSORCIO INTER-PROVINCIAL                       │
├─────────────────────────────────────────────────────────────────────┤
""")

total_citizens = len(df_citizens)
total_vulnerable = (df_citizens['zone_vulnerability_index'] > 0.5).sum()
total_assisted = df_citizens['social_assistance'].sum()
avg_demand = df_citizens['service_demand_score'].mean()

print(f"│  Ciudadanos registrados:     {total_citizens:>10,}                        │")
print(f"│  Poblacion vulnerable:       {total_vulnerable:>10,} ({total_vulnerable/total_citizens*100:.1f}%)               │")
print(f"│  Beneficiarios asistencia:   {int(total_assisted):>10,} ({total_assisted/total_citizens*100:.1f}%)               │")
print(f"│  Score demanda promedio:     {avg_demand:>10.2f}                          │")
print(f"│  Precision del modelo:       {r2*100:>10.1f}%                          │")

print("""
├─────────────────────────────────────────────────────────────────────┤
│                      POR PROVINCIA                                   │
├─────────────────────────────────────────────────────────────────────┤""")

for prov in GOVERNMENT_CONFIG['provinces']:
    prov_data = df_citizens[df_citizens['province'] == prov['name']]
    vuln_pct = (prov_data['zone_vulnerability_index'] > 0.5).mean() * 100
    coverage = df_allocation[df_allocation['province'] == prov['name']]['coverage'].values[0]
    print(f"│  {prov['name']:<15} Vuln: {vuln_pct:>5.1f}%  Cobertura: {coverage:>5.1f}%          │")

print("└─────────────────────────────────────────────────────────────────────┘")

## 8. Resumen y Garantias

In [ ]:
print("RESUMEN DEL DEMO GOVERNMENT")
print("=" * 60)

print("""
GARANTIAS DE PRIVACIDAD Y SEGURIDAD
-----------------------------------
 Datos de ciudadanos NUNCA compartidos en plaintext
 Cumplimiento Ley 25.326 de Proteccion de Datos
 Encriptacion end-to-end con CKKS (128-bit)
 Solo estadisticas agregadas son compartidas
 Cada provincia mantiene soberania de sus datos
 Audit trail inmutable en blockchain

BENEFICIOS DEL CONSORCIO
------------------------
 Mejor asignacion de recursos entre provincias
 Deteccion temprana de poblaciones vulnerables
 Optimizacion del gasto publico
 Politicas basadas en evidencia
 Transparencia en la toma de decisiones

METRICAS DEL MODELO
-------------------
""")
print(f"  Provincias participantes: {len(GOVERNMENT_CONFIG['provinces'])}")
print(f"  Total ciudadanos: {total_citizens:,}")
print(f"  Segmentos identificados: {n_clusters}")
print(f"  Silhouette score: {silhouette:.4f}")
print(f"  R2 Score (prediccion): {r2:.4f}")
print(f"  MAE: {mae:.4f}")

---

## Proximos Pasos

1. **Integrar con SINEP/SINTyS**: Conexion con sistemas nacionales
2. **Expandir a mas provincias**: Cobertura nacional
3. **Agregar mas servicios**: Transporte, seguridad, etc.

**Documentacion**: https://apifhe.xcapit.com/api/v2/docs/